I. SIESTA

SIESTA is a linear combination of atomic orbitals (LCAO) Density Functional Theory (DFT) code that employs norm‑conserving pseudopotentials and highly efficient numerical atomic basis sets. In this tutorial, all calculations were carried out using SIESTA 5.4.2 together with PseudoDojo-curated pseudopotentials to ensure consistency and high transferability.

For the electronic structure calculations, we used the standard double‑zeta polarized (DZP) basis set distributed with SIESTA, which provides a good balance between accuracy and computational cost for typical materials simulations.

The Atomistic Simulation Environment (ASE) served as the workflow manager, handling input generation, job execution, and post‑processing. Instead of relying on SIESTA’s built‑in geometry optimization routines, all structural relaxations were performed using ASE’s optimizers, which offer more flexibility and tighter integration with Python-based workflows.

Single‑point calculation performed with SIESTA through ASE introduces the essential structure of a SIESTA workflow: defining the atomic system, selecting the basis set and pseudopotentials, configuring the calculator, and executing the run to obtain the electronic ground‑state properties without performing any geometry optimization.

In [ ]:
from ase import Atoms
from ase.build import bulk
from ase.calculators.siesta import Siesta
from ase.units import Ry
from ase.visualize import view
import os


#create a new directory for the calculation

os.mkdir("c2_siesta")
os.chdir("c2_siesta")

# use ASE to create a diamond structure of carbon

atoms = bulk('C', 'diamond', a=3.567, cubic=True)

# Minimal Siesta calculator

atoms.calc = Siesta(label='c2', # system label
                        xc='PBE', # exchange-correlation functional
                        pseudo_path=os.environ["SIESTA_PSEUDO_DIR"], #path to pseudopotentials
                        pseudo_qualifier='', #in necessary, specify the qualifier for the pseudopotentials
                        symlink_pseudos=True,
                        mesh_cutoff=200 * Ry, # cutoff for the real-space grid
                        energy_shift=0.01 * Ry,
                        basis_set='DZP', # basis set
                        spin='non-polarized', # spin configuration
                        kpts=(4, 4, 4), # k-point grid
                        fdf_arguments={'DM.MixingWeight': 0.1, 'MaxSCFIterations': 100}, # additional arguments for the SIESTA input file
               )

energy = atoms.get_total_energy() # execute the calculation and get the total energy

os.chdir("../")

print("Energy:", energy)

view(atoms, viewer='x3d') # atomistic visualization of the structure using x3d viewer in IPython



2. Converging the k‑point mesh in SIESTA is a critical prerequisite for obtaining reliable and reproducible results. The required k‑point density depends on the Bravais lattice of the system and scales with the reciprocal‑space dimensions of the periodic simulation cell. In practice, this means that larger real‑space cells require fewer k‑points, while smaller cells demand a denser mesh. The appropriate k‑point grid must be determined specifically for the system under study. It cannot—and should not—be copied from another calculation, even if the material appears similar. Each structure has its own reciprocal‑space characteristics, and the optimal grid is therefore unique.A standard way to determine the correct mesh is to converge the total energy with respect to the k‑point sampling. By systematically increasing the grid density and monitoring how the total energy changes, one can identify the smallest mesh that yields stable, well‑converged results. This procedure ensures that all subsequent calculations—geometry optimizations, electronic structure analysis, or property evaluations—are built on a solid numerical foundation.

In [ ]:
from ase import Atoms
from ase.calculators.siesta import Siesta
import os
from ase.build import bulk
from ase.units import Ry
import numpy as np
import matplotlib.pyplot as plt

os.mkdir("c2_siesta_kpts")
os.chdir("c2_siesta_kpts")

atoms = bulk('C', 'diamond', a=3.567, cubic=True)

kk_array = []
energy_array = []

for kk in range(1, 9, 1): # loop over k-points from 1x1x1 to 8x8x8
 
    atoms.calc = Siesta(label='c2',
                            xc='PBE',
                            pseudo_path=os.environ["SIESTA_PSEUDO_DIR"],
                            pseudo_qualifier='',
                            symlink_pseudos=True,
                            mesh_cutoff=200 * Ry,
                            energy_shift=0.01 * Ry,
                            basis_set='DZP',
                            spin='non-polarized',
                            kpts=(kk, kk, kk),
                            fdf_arguments={'DM.MixingWeight': 0.1, 'MaxSCFIterations': 100},
                )
    energy = atoms.get_total_energy()
    print("k-points:", kk)
    kk_array.append(kk)
    print("Energy:", energy)
    energy_array.append(energy)

# Plot k-point energy convergence

plt.figure(figsize=(10, 6))  # Set the figure size (optional)
plt.plot(kk_array, energy_array, marker='o', linestyle='-', color='b')  # Plot the data
plt.title('Energy vs k-points')  # Add a title to the plot
plt.xlabel('k-points')  # Label for the x-axis
plt.ylabel('Energy')  # Label for the y-axis
plt.grid(True)  # Add a grid (optional)
plt.show()  # Display the plot

os.chdir("../")


3. SIESTA evaluates the electronic density and related quantities on a real‑space integration grid. The quality of this grid has a direct impact on the numerical stability of the calculation. A fine real‑space grid leads to well‑converged energies and forces, ensuring smooth potential surfaces and reliable structural optimization. In contrast, an insufficiently dense grid can introduce numerical artifacts known as the eggbox effect, where the total energy and forces fluctuate as atoms move relative to the underlying grid. These fluctuations degrade the accuracy of geometry optimizations and any properties derived from them. For this reason, converging the real‑space grid (via the MeshCutoff parameter) is considered good practice. By systematically increasing the grid cutoff and monitoring the stability of the total energy and forces, one can identify a cutoff value that eliminates eggbox artifacts and provides consistent, physically meaningful results for the system under study.

In [ ]:
from ase import Atoms
from ase.calculators.siesta import Siesta
from ase.build import bulk
import os
from ase.units import Ry
import numpy as np
import matplotlib.pyplot as plt

os.mkdir("c2_siesta_cutoff")
os.chdir("c2_siesta_cutoff")

atoms = bulk('C', 'diamond', a=3.567, cubic=True)

kk_array = []
energy_array = []

for kk in range(150, 260, 10):
    # Minimal Vasp calculator for testing
    atoms.calc = Siesta(label='c2',
                            xc='PBE',
                            pseudo_path=os.environ["SIESTA_PSEUDO_DIR"],
                            pseudo_qualifier='',
                            symlink_pseudos=True,
                            mesh_cutoff=kk * Ry,
                            energy_shift=0.01 * Ry,
                            basis_set='DZP',
                            spin='non-polarized',
                            kpts=(5, 5, 5),
                            fdf_arguments={'DM.MixingWeight': 0.1, 'MaxSCFIterations': 100},
                )
    
    energy = atoms.get_total_energy()
    print("k-points:", kk)
    kk_array.append(kk)
    print("Energy:", energy)
    energy_array.append(energy)
# Run a single-point calculation

plt.figure(figsize=(10, 6))  # Set the figure size (optional)
plt.plot(kk_array, energy_array, marker='o', linestyle='-', color='b')  # Plot the data
plt.title('Energy vs real space mesh')  # Add a title to the plot
plt.xlabel('kk')  # Label for the x-axis
plt.ylabel('real space mesh')  # Label for the y-axis
plt.grid(True)  # Add a grid (optional)
plt.show()  # Display the plot

os.chdir("../")


4. Geometry and lattice optimization in this tutorial are performed using the ASE optimization algorithms, while SIESTA is used strictly as a force and energy calculator. This separation of roles is intentional. Many DFT packages in materials science—including SIESTA—provide basic geometry optimizers that are functional but often limited in robustness, flexibility, and convergence behavior. In contrast, external frameworks such as ASE offer modern, well‑tested optimization algorithms that integrate smoothly with Python workflows and provide more reliable convergence for both atomic positions and cell degrees of freedom. The optimizer used here is the Broyden–Fletcher–Goldfarb–Shanno (BFGS) algorithm. BFGS is a quasi‑Newton method that combines gradient information with an evolving approximation of the inverse Hessian. This allows it to accelerate convergence by effectively estimating curvature without performing expensive second‑derivative calculations. For comparison, SIESTA’s built‑in Conjugate Gradient (CG) optimizer relies solely on first‑order information. CG constructs search directions that are conjugate with respect to the Hessian but never stores or approximates the Hessian itself. While efficient, CG typically converges more slowly and less smoothly than quasi‑Newton methods, especially for complex materials or systems with soft modes. To enable full lattice optimization, ASE provides the UnitCellFilter class, which exposes the cell degrees of freedom to the optimizer. When wrapped around the atomic configuration, UnitCellFilter ensures that both atomic positions and lattice vectors are updated consistently during the optimization process.

In [ ]:
from ase import Atoms
from ase.calculators.siesta import Siesta
from ase.units import Ry
from ase.build import bulk
from ase.visualize import view
import os
from ase.optimize.bfgs import BFGS
from ase.filters import UnitCellFilter
from ase.io import write


os.mkdir("c2_siesta_opt")
os.chdir("c2_siesta_opt")

atoms = bulk('C', 'diamond', a=3.567, cubic=True)

print("Initial cell:")
print(atoms.cell) # print the initial cell parameters before the optimization

# Minimal Siesta calculator for testing
atoms.calc = Siesta(label='c2',
                        xc='PBE',
                        pseudo_path=os.environ["SIESTA_PSEUDO_DIR"],
                        pseudo_qualifier='',
                        symlink_pseudos=True,
                        mesh_cutoff=230 * Ry,
                        energy_shift=0.01 * Ry,
                        basis_set='DZP',
                        spin='non-polarized',
                        kpts=(5, 5, 5),
                        fdf_arguments={'DM.MixingWeight': 0.1, 'MaxSCFIterations': 100},
               )

energy = atoms.get_total_energy()

ucf = UnitCellFilter(atoms) # set up a unit cell filter to allow optimization of the cell parameters
opt = BFGS(ucf, trajectory='cellopt.traj') # set up a BFGS optimizer to optimize the cell parameters of the structure
opt.run(fmax=0.01) # converge the optimization until the maximum force is below 0.01 eV/Å

# Run a single-point calculation

os.chdir("../")

print("Final cell:")
print(atoms.cell) # print the optimized cell parameters after the optimization is complete

write('cell_siesta.traj', atoms) # the optimization trajectory is saved as a set of geometry snapshots in a .traj file, which can be visualized later using ASE or other visualization tools

view(atoms, viewer='x3d')

5. The Density of States (DOS) is one of the fundamental electronic properties computed in Density Functional Theory. It describes how many electronic states are available at each energy level and provides direct insight into the material’s electronic behavior. By examining the DOS, we can identify the valence band, conduction band, and the band gap, which together determine whether a material behaves as an insulator, semiconductor, or metal. Because of this, DOS analysis is an essential step in interpreting and understanding the electronic structure obtained from DFT calculations.

In [ ]:
from ase import Atoms
from ase.calculators.siesta import Siesta
from ase.units import Ry
import os
from ase.io import read
from ase.dft.dos import DOS # ASE calss for calculating the density of states (DOS)
import matplotlib.pyplot as plt

atoms = read('cell_siesta.traj')
atoms.pbc=True

os.mkdir("c2_siesta_dos")
os.chdir("c2_siesta_dos")

atoms.calc = Siesta(label='c2',
                        xc='PBE',
                        pseudo_path=os.environ["SIESTA_PSEUDO_DIR"],
                        pseudo_qualifier='',
                        symlink_pseudos=True,
                        mesh_cutoff=230 * Ry,
                        energy_shift=0.01 * Ry,
                        basis_set='DZP',
                        spin='non-polarized',
                        kpts=(5, 5, 5),
                        fdf_arguments={'DM.MixingWeight': 0.1, 'MaxSCFIterations': 100, 'WriteEigenvalues': True, 'SaveHS': True,},
               )

energy = atoms.get_potential_energy()

dos = DOS(atoms.calc,
          width=0.05,
          npts=3000)

E = dos.get_energies()
D = dos.get_dos()

os.chdir("../")

# Select only -10 to +10 eV around Ef
mask = (E >= -10) & (E <= 10)

plt.plot(E[mask], D[mask])
plt.axvline(0, color='k', linestyle='--')
plt.xlim(-10, 10)
plt.xlabel(r'$E - E_F$ (eV)')
plt.ylabel('DOS (states/eV)')
plt.show()


6. While bulk calculations are essential for understanding intrinsic material properties, many applications in catalysis, electronics, and nanotechnology require detailed knowledge of surface properties. In periodic DFT codes such as SIESTA, surfaces are modeled using slab geometries, where a finite number of atomic layers represent the surface and vacuum is added to isolate it from periodic images. ASE provides convenient tools for constructing such slabs. Using its built‑in ase.build.surface function, one can cleave a bulk crystal along a chosen set of Miller indices. In this tutorial, we generate the diamond (100) surface by slicing the bulk diamond structure along the (1 0 0) plane and adding sufficient vacuum to avoid interactions between repeated slabs. Once the slab is constructed, the geometry optimization is applied only to the atomic positions, not to the cell parameters. The lattice constants are optimized at the bulk level and then kept fixed during surface relaxation. This ensures that the slab inherits physically meaningful bulk geometry while allowing the surface layers to relax into their energetically preferred configuration.

In [ ]:
from ase import Atoms
from ase.calculators.siesta import Siesta
from ase.units import Ry
from ase.build import surface
from ase.visualize import view
from ase.io import read
from ase.io import write
import os
from ase.optimize.bfgs import BFGS

atoms = read('cell_siesta.traj')
atoms.pbc=True

slab = surface(atoms, (1,0,0), 2, vacuum=10.0) # create a slab of the optimized structure with 4 layers and 15 Å of vacuum
slab.pbc=True

# center slab in vacuum
slab.center(axis=2)


os.mkdir("c2_siesta_slab_opt")
os.chdir("c2_siesta_slab_opt")

# Minimal Vasp calculator for testing
slab.calc = Siesta(label='c2',
                        xc='PBE',
                        pseudo_path=os.environ["SIESTA_PSEUDO_DIR"],
                        pseudo_qualifier='',
                        symlink_pseudos=True,
                        mesh_cutoff=230 * Ry,
                        energy_shift=0.01 * Ry,
                        basis_set='DZP',
                        spin='non-polarized',
                        kpts=(5, 5, 1),
                        fdf_arguments={'DM.MixingWeight': 0.1, 'MaxSCFIterations': 100},
               )

energy = slab.get_total_energy()

opt = BFGS(slab, trajectory='cellopt.traj')
opt.run(fmax=0.03)

# Run a single-point calculation

os.chdir("../")

write('slab_siesta.traj', slab)


view(slab, viewer='x3d')

7. The surface Density of States (DOS) provides essential insight into how a surface modifies the electronic structure of a material. Unlike the bulk, where all bonds are fully coordinated, a surface exposes atoms with unsatisfied valence—leading to electronic features that can differ dramatically from the bulk behavior. The DOS of the diamond (100) surface illustrates this clearly. Although bulk diamond is a wide‑band‑gap insulator, the (100) surface shows no band gap and exhibits metallic character. At first glance this result may seem counterintuitive. However, the explanation lies in the orbital configuration of the surface atoms. The top‑layer carbon atoms possess dangling bonds that are not compensated by neighboring atoms. These dangling‑bond states fall within the band gap of bulk diamond and create partially filled surface states, giving rise to metallic behavior. This metallicity is not just a computational artifact—it is consistent with experimental observations, where clean diamond surfaces often display surface conductivity due to these unsaturated bonds. Understanding this effect is crucial when studying surface chemistry, adsorption, catalysis, or electronic devices based on diamond surfaces.

In [ ]:
from ase import Atoms
from ase.calculators.siesta import Siesta
from ase.units import Ry
import os
from ase.io import read
from ase.dft.dos import DOS
import matplotlib.pyplot as plt

atoms = read('slab_siesta.traj')
atoms.pbc=True

os.mkdir("c2_siesta_dos_slab")
os.chdir("c2_siesta_dos_slab")

atoms.calc = Siesta(label='c2',
                        xc='PBE',
                        pseudo_path=os.environ["SIESTA_PSEUDO_DIR"],
                        pseudo_qualifier='',
                        symlink_pseudos=True,
                        mesh_cutoff=230 * Ry,
                        energy_shift=0.01 * Ry,
                        basis_set='DZP',
                        spin='non-polarized',
                        kpts=(5, 5, 1),
                        fdf_arguments={'DM.MixingWeight': 0.1, 'MaxSCFIterations': 100, 'WriteEigenvalues': True, 'SaveHS': True,},
               )

energy = atoms.get_potential_energy()

dos = DOS(atoms.calc,
          width=0.05,
          npts=3000)

E = dos.get_energies()
D = dos.get_dos()

os.chdir("../")

# Select only -10 to +10 eV around Ef
mask = (E >= -10) & (E <= 10)

plt.plot(E[mask], D[mask])
plt.axvline(0, color='k', linestyle='--')
plt.xlim(-10, 10)
plt.xlabel(r'$E - E_F$ (eV)')
plt.ylabel('DOS (states/eV)')
plt.show()


8. In reality, a pristine diamond surface exists only under ultrahigh‑vacuum (UHV) conditions and even then only for short periods of time. Under ambient or experimental conditions, the surface rapidly becomes passivated, most commonly by hydrogen atoms. To model realistic surfaces, we therefore construct a hydrogen‑terminated diamond (100) slab and optimize its geometry again. ASE provides convenient tools for building such passivated surfaces. First, we identify the top and bottom surface carbon atoms, which are the ones carrying dangling bonds. Hydrogen atoms are then added to these sites to saturate the dangling bonds and restore the insulating character of the surface. To avoid artificial interactions between hydrogens, we arrange them in a pattern that maximizes the distance between neighboring H atoms, ensuring a physically meaningful termination. After adding the hydrogens, the slab is relaxed again, allowing both the surface carbon atoms and the hydrogen atoms to settle into their energetically preferred configuration. This passivated surface serves as a more realistic model for studying electronic properties, adsorption, and surface chemistry under typical experimental conditions.

In [ ]:
import numpy as np
from ase.io import read
from ase.build import surface
from ase import Atom
from ase.visualize import view


atoms = read('cell_siesta.traj')
atoms.pbc = True

slab = surface(atoms, (1,0,0), 2, vacuum=10.0)
slab.center(axis=2)

# Highest carbon z coordinate
zmax = max(atom.position[2] for atom in slab if atom.symbol == 'C')

# Select top-layer carbon atoms
tol = 0.5  # Å tolerance
top_carbons = [i for i, atom in enumerate(slab)
               if atom.symbol == 'C' and zmax - atom.position[2] < tol]

CH = 0.8


pos = slab[top_carbons[0]].position + np.array([-0.5, -0.5, CH])
slab.append(Atom('H', pos))

pos = slab[top_carbons[0]].position + np.array([0.5, 0.5, CH])
slab.append(Atom('H', pos))

pos = slab[top_carbons[1]].position + np.array([-0.5, 0.5, CH])
slab.append(Atom('H', pos))

pos = slab[top_carbons[1]].position + np.array([0.5, -0.5, CH])
slab.append(Atom('H', pos))


# Highest carbon z coordinate
zmin = min(atom.position[2] for atom in slab if atom.symbol == 'C')

top_carbons = [i for i, atom in enumerate(slab)
               if atom.symbol == 'C' and atom.position[2] - zmin < tol]


pos = slab[top_carbons[0]].position - np.array([-0.5, -0.5, CH])
slab.append(Atom('H', pos))

pos = slab[top_carbons[0]].position - np.array([0.5, 0.5, CH])
slab.append(Atom('H', pos))

pos = slab[top_carbons[1]].position - np.array([-0.5, 0.5, CH])
slab.append(Atom('H', pos))

pos = slab[top_carbons[1]].position - np.array([0.5, -0.5, CH])
slab.append(Atom('H', pos))

view(slab, viewer='x3d')

9. Self‑consistent field (SCF) convergence and geometry relaxation in LCAO‑based DFT codes such as SIESTA depend strongly on the quality of the initial atomic configuration. When the starting geometry is far from the true minimum, the SCF cycle may become unstable, stall, or fail to converge entirely. This is exactly what happens for the H‑passivated diamond (100) surface: the initial guess contains significant strain and unfavorable bond orientations, making the SCF procedure overly sensitive. A practical and widely used strategy in SIESTA is to temporarily loosen the computational settings to help the optimizer explore a broader region of configuration space. Reducing the basis‑set size, lowering the k‑point density, and decreasing the real‑space mesh cutoff all make the SCF cycle more forgiving. Although these settings are not suitable for final production calculations, they allow the geometry optimizer to move the system closer to a physically meaningful structure where full‑accuracy parameters can later be restored. In this tutorial, we apply this approach by reducing the k‑point mesh to 3×3×3 and lowering the real‑space grid cutoff to 170 Ry. With these relaxed settings, the SCF converges reliably, enabling the geometry optimizer to correct the initial distortions. Once the structure is sufficiently improved, the calculation can be repeated with fully converged parameters to obtain accurate energies and electronic properties.

In [ ]:
import numpy as np
from ase.io import read
from ase.io import write
from ase.build import surface
from ase import Atom
from ase.visualize import view
from ase.calculators.siesta import Siesta
from ase.units import Ry
from ase.optimize.bfgs import BFGS
import os


atoms = read('cell_siesta.traj')
atoms.pbc = True

slab = surface(atoms, (1,0,0), 2, vacuum=10.0)
slab.center(axis=2)

# Highest carbon z coordinate
zmax = max(atom.position[2] for atom in slab if atom.symbol == 'C')

# Select top-layer carbon atoms
tol = 0.5  # Å tolerance
top_carbons = [i for i, atom in enumerate(slab)
               if atom.symbol == 'C' and zmax - atom.position[2] < tol]

# Add H atoms 1.09 Å above each top carbon
CH = 0.8


pos = slab[top_carbons[0]].position + np.array([-0.5, -0.5, CH])
slab.append(Atom('H', pos))

pos = slab[top_carbons[0]].position + np.array([0.5, 0.5, CH])
slab.append(Atom('H', pos))

pos = slab[top_carbons[1]].position + np.array([-0.5, 0.5, CH])
slab.append(Atom('H', pos))

pos = slab[top_carbons[1]].position + np.array([0.5, -0.5, CH])
slab.append(Atom('H', pos))


# Highest carbon z coordinate
zmin = min(atom.position[2] for atom in slab if atom.symbol == 'C')

top_carbons = [i for i, atom in enumerate(slab)
               if atom.symbol == 'C' and atom.position[2] - zmin < tol]


pos = slab[top_carbons[0]].position - np.array([-0.5, -0.5, CH])
slab.append(Atom('H', pos))

pos = slab[top_carbons[0]].position - np.array([0.5, 0.5, CH])
slab.append(Atom('H', pos))

pos = slab[top_carbons[1]].position - np.array([-0.5, 0.5, CH])
slab.append(Atom('H', pos))

pos = slab[top_carbons[1]].position - np.array([0.5, -0.5, CH])
slab.append(Atom('H', pos))

os.mkdir("c2_siesta_slab_h_opt")
os.chdir("c2_siesta_slab_h_opt")

# Minimal Vasp calculator for testing
slab.calc = Siesta(label='c2',
                        xc='PBE',
                        pseudo_path=os.environ["SIESTA_PSEUDO_DIR"],
                        pseudo_qualifier='',
                        symlink_pseudos=True,
                        mesh_cutoff=170 * Ry,
                        energy_shift=0.01 * Ry,
                        basis_set='SZP',
                        spin='non-polarized',
                        kpts=(3, 3, 1),
                        fdf_arguments={'DM.MixingWeight': 0.1, 'MaxSCFIterations': 100},
               )

energy = slab.get_total_energy()

opt = BFGS(slab, trajectory='cellopt.traj')
opt.run(fmax=0.03)

# Run a single-point calculation

os.chdir("../")

write('slab_h_siesta.traj', slab)


view(slab, viewer='x3d')

10. The fully optimized geometry differs substantially from our initial guess, confirming that the original configuration was far from the true minimum. After relaxing the structure with the loosened computational settings, we restored the high‑accuracy parameters. At this stage, the system was already close to equilibrium, and the SCF cycle was stable. As a result, the final high‑precision geometry optimization converged in only a few steps, demonstrating the effectiveness of the two‑stage relaxation strategy.

In [ ]:
import numpy as np
from ase.io import read
from ase.io import write
from ase import Atom
from ase.visualize import view
from ase.calculators.siesta import Siesta
from ase.units import Ry
from ase.optimize.bfgs import BFGS
import os


atoms = read('slab_h_siesta.traj')
atoms.pbc = True

os.mkdir("c2_siesta_slab_h_opt_high")
os.chdir("c2_siesta_slab_h_opt_high")

# Minimal Vasp calculator for testing
slab.calc = Siesta(label='c2',
                        xc='PBE',
                        pseudo_path=os.environ["SIESTA_PSEUDO_DIR"],
                        pseudo_qualifier='',
                        symlink_pseudos=True,
                        mesh_cutoff=230 * Ry,
                        energy_shift=0.01 * Ry,
                        basis_set='DZP',
                        spin='non-polarized',
                        kpts=(5, 5, 1),
                        fdf_arguments={'DM.MixingWeight': 0.1, 'MaxSCFIterations': 100},
               )

energy = slab.get_total_energy()

opt = BFGS(slab, trajectory='cellopt.traj')
opt.run(fmax=0.03)

# Run a single-point calculation

os.chdir("../")

write('slab_h_siesta_high.traj', slab)


view(slab, viewer='x3d')

11. The Density of States (DOS) of the hydrogen‑passivated diamond (100) surface shows a large band gap, closely matching that of bulk diamond. This behavior is expected: once the surface carbon atoms are saturated with hydrogen, the dangling‑bond states responsible for the metallic character of the pristine surface are removed. With these mid‑gap states eliminated, the electronic structure of the slab returns to that of an insulator, demonstrating that hydrogen termination effectively restores the bulk‑like electronic properties at the surface.

In [ ]:
from ase import Atoms
from ase.calculators.siesta import Siesta
from ase.units import Ry
import os
from ase.io import read
from ase.dft.dos import DOS
import matplotlib.pyplot as plt

atoms = read('slab_h_siesta_high.traj')
atoms.pbc=True

os.mkdir("c2_siesta_dos_slab_h")
os.chdir("c2_siesta_dos_slab_h")

atoms.calc = Siesta(label='c2',
                        xc='PBE',
                        pseudo_path=os.environ["SIESTA_PSEUDO_DIR"],
                        pseudo_qualifier='',
                        symlink_pseudos=True,
                        mesh_cutoff=230 * Ry,
                        energy_shift=0.01 * Ry,
                        basis_set='DZP',
                        spin='non-polarized',
                        kpts=(5, 5, 1),
                        fdf_arguments={'DM.MixingWeight': 0.1, 'MaxSCFIterations': 100, 'WriteEigenvalues': True, 'SaveHS': True,},
               )

energy = atoms.get_potential_energy()

dos = DOS(atoms.calc,
          width=0.05,
          npts=3000)

E = dos.get_energies()
D = dos.get_dos()

os.chdir("../")

# Select only -10 to +10 eV around Ef
mask = (E >= -10) & (E <= 10)

plt.plot(E[mask], D[mask])
plt.axvline(0, color='k', linestyle='--')
plt.xlim(-10, 10)
plt.xlabel(r'$E - E_F$ (eV)')
plt.ylabel('DOS (states/eV)')
plt.show()


12. The Density of States (DOS) tells us how electronic states are distributed relative to the Fermi level, but it does not provide the absolute energy of the Fermi level. In periodic DFT calculations, all eigenvalues are referenced to an arbitrary internal zero, determined by the basis set, pseudopotentials, and numerical setup. As a result, absolute energy levels cannot be compared directly between two different materials or even between two separate calculations of the same material using different computational parameters. To obtain meaningful, comparable energy references, we compute the work function or, more precisely, the ionization potential of the slab. This quantity gives the energy of the valence‑band maximum (VBM) relative to the vacuum level, which is the same physical reference for every slab. By anchoring the electronic structure to the vacuum level, we can compare band edges, Fermi levels, and work functions across different systems in a consistent way. To perform work‑function or ionization‑potential calculations, the slab must include a sufficiently thick vacuum region. The electrostatic potential must reach a flat plateau in the vacuum, indicating that the potential is constant and free from interactions with the slab. Once this plateau is identified, we extract the vacuum electrostatic potential and reference the VBM (or Fermi level) to it. SIESTA outputs the electrostatic potential on a real‑space grid in the file ElectrostaticPotential.grid.nc, which contains the volumetric data needed to compute the averaged potential along the slab normal. To generate this file, several options must be enabled in the SIESTA input: 'WriteEigenvalues': True,
'SaveHS': True, 'SaveElectrostaticPotential': True, 'WriteVH': True,


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re
from ase import Atoms
from ase.calculators.siesta import Siesta
from ase.units import Ry
import os
from ase.io import read

slab = read('slab_h_siesta_high.traj')
slab.pbc=True

os.mkdir("c2_siesta_wf_slab_h")
os.chdir("c2_siesta_wf_slab_h")


slab.calc = Siesta(label='slab',
                        xc='PBE',
                        pseudo_path=os.environ["SIESTA_PSEUDO_DIR"],
                        pseudo_qualifier='',
                        symlink_pseudos=True,
                        mesh_cutoff=230 * Ry,
                        energy_shift=0.01 * Ry,
                        basis_set='DZP',
                        spin='non-polarized',
                        kpts=(5, 5, 1),
                        fdf_arguments={'DM.MixingWeight': 0.1, 'MaxSCFIterations': 100, 'WriteEigenvalues': True, 'SaveHS': True,'SaveElectrostaticPotential': True, 'WriteVH': True,},
               )


energy = slab.get_potential_energy()

os.chdir("../")


The ionization potential of the hydrogen‑passivated diamond (100) slab is calculated to be 2.584207 eV, whereas the pristine diamond (100) surface exhibits a much higher value of 6.660054 eV. This dramatic difference is well‑known experimentally and highlights the crucial role of surface reconstruction and surface termination in determining electronic properties. Hydrogen termination fundamentally alters the surface electronic environment. By saturating the dangling bonds on the topmost carbon atoms, hydrogen atoms create a surface dipole layer that shifts the electrostatic potential downward. This dipole makes it energetically easier to remove an electron from the surface, resulting in a much lower ionization potential. In practical terms, the H‑terminated surface becomes far more favorable for electron emission, a property exploited in applications such as negative‑electron‑affinity (NEA) diamond devices. In contrast, the pristine diamond (100) surface lacks these stabilizing dipoles. Its unsaturated dangling bonds produce mid‑gap states and a higher surface potential, making electron extraction significantly more difficult. The high ionization potential reflects this unfavorable electronic environment. These results demonstrate how surface chemistry directly controls the electronic structure, and they emphasize the importance of accurate surface modeling when studying real materials or designing diamond‑based electronic and optoelectronic devices.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re
from ase import Atoms
from ase.calculators.siesta import Siesta
from ase.units import Ry
import os
from ase.io import read
import numpy as np
import matplotlib.pyplot as plt



slab = read('slab_siesta.traj')
slab.pbc=True

os.mkdir("c2_siesta_wf_slab_h_non")
os.chdir("c2_siesta_wf_slab_h_non")


slab.calc = Siesta(label='slab',
                        xc='PBE',
                        pseudo_path=os.environ["SIESTA_PSEUDO_DIR"],
                        pseudo_qualifier='',
                        symlink_pseudos=True,
                        mesh_cutoff=230 * Ry,
                        energy_shift=0.01 * Ry,
                        basis_set='DZP',
                        spin='non-polarized',
                        kpts=(5, 5, 1),
                        fdf_arguments={'DM.MixingWeight': 0.1, 'MaxSCFIterations': 100, 'WriteEigenvalues': True, 'SaveHS': True,'SaveElectrostaticPotential': True, 'WriteVH': True,},
               )


energy = slab.get_potential_energy()


os.chdir("../")
